In [112]:
import os
import random
import argparse

import dill
import torch
import gdown
import numpy as np
import pandas as pd
from torch.autograd import Variable
from torch.nn.utils.rnn import pad_sequence

In [ ]:
def is_in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False
    
if is_in_colab():
    gdown.download_folder("https://drive.google.com/drive/folders/1Xo5K-fP6hCjqfm2_8U99VdWOmAnPltQo?usp=drive_link")

In [ ]:
if not is_in_colab():
    args = argparse.Namespace(
        data="./data/ICEWS14_forecasting",
        dataset="ICEWS14_forecasting",
        epochs_conv=100,
        weight_decay_conv=0.000001,
        embedding_size=200,
        lr=0.0001
    )
else:
    args = argparse.Namespace(
        data="./ILR-IR/data/ICEWS14_forecasting",
        dataset="ICEWS14_forecasting",
        epochs_conv=100,
        weight_decay_conv=0.000001,
        embedding_size=200,
        lr=0.0001
    ) 

def load_data(args):
    with open(os.path.join('{}'.format(args.data), 'stat.txt'), 'r') as fr:
        for line in fr:
            line_split = line.split()
            num_e, num_r = int(line_split[0]), int(line_split[1])

    relation_embeddings = np.random.randn(num_r * 2, args.embedding_size)
    print("Initialised relations and entities randomly")
    return num_e, num_r, torch.FloatTensor(relation_embeddings)

num_e, num_r, relation_embeddings = load_data(args)

Initialised relations and entities randomly


In [101]:
def build_data(path, num_r):
    t_quads = {}
    t_quads_re = {}

    all_triples = set()
    quads_id = {}
    with open(os.path.join(path, 'data.txt'), 'r') as fr:
        times = set()
        for i, line in enumerate(fr):
            line_split = line.split()
            time = int(line_split[3])
            times.add(time)

            e1, relation, e2 = int(line_split[0]), int(
                line_split[1]), int(line_split[2])

            all_triples.add((e1, relation, e2))
            all_triples.add((e2, relation+num_r, e1))

            t_quads.setdefault(time, []).append((e1, relation, e2))
            t_quads_re.setdefault(time, []).append((e2, relation+num_r, e1))

        all_triples = list(all_triples)
        for i, triple in enumerate(all_triples):
            quads_id[triple] = i

    all_times = list(times)
    all_times.sort()

    t_quadid = {}
    t_quadid_re = {}
    for t in all_times:
        for i in t_quads[t]:
            t_quadid.setdefault(t, []).append(quads_id[i])
        for j in t_quads_re[t]:
            t_quadid_re.setdefault(t, []).append(quads_id[j])
    print("number of triples ->", len(all_triples))

    return t_quadid, t_quadid_re, all_triples, all_times

_, _, all_triples, _ = build_data(args.data, num_r)

number of triples -> 100590


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence

import math

config = {
    "data": {
        "data_dir": "./data/ICEWS14_forecasting",
        "dataset": "ICEWS14_forecasting"
    },
    "training": {
        "state": "train",  # "train" or "test"
        "epochs_conv": 100,
        "lr": 0.0001,
        "weight_decay_conv": 0.000001
    },
    "model": {
        "embedding_size": 200
    },
    "temporal": {
        "neg_ratio": 1
    },
    "device": {
        "device_type": "cpu"  # cpu, cuda
    },
    "experiment": {
        "save_models": True,
        "results_dir": "./results/bestmodel",
        "log_level": "INFO"
    },
    "ds_specific": {
        "ICEWS14_forecasting": {
            "his_len": 13
        },
        "ICEWS18": {
            "his_len": 10
        },
        "ICEWS0515_forecasting": {
            "his_len": 150
        }
    }
}


DEVICE = config["device"]["device_type"]
HAS_ACCELERATION = DEVICE == 'cuda'


class RelTemporalEncoding(nn.Module):
    def __init__(self, n_hid, max_len=4020, dropout=0.2):
        super(RelTemporalEncoding, self).__init__()
        position = torch.arange(0., max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, n_hid, 2) *
                             -(math.log(10000.0) / n_hid))
        emb = nn.Embedding(max_len, n_hid)
        emb.weight.data[:, 0::2] = torch.sin(
            position * div_term) / math.sqrt(n_hid)
        emb.weight.data[:, 1::2] = torch.cos(
            position * div_term) / math.sqrt(n_hid)
        emb.requires_grad_(False)
        self.emb = emb
        self.lin = nn.Linear(n_hid, n_hid)

    def forward(self, x, t):
        return x + self.lin(self.emb(t))


class TypeGAT(nn.Module):
    def __init__(self, num_e, num_r, relation_embeddings, out_dim):
        super(TypeGAT, self).__init__()

        self.num_e = num_e
        self.num_r = num_r
        self.in_dim = relation_embeddings.shape[1]
        self.out_dim = out_dim

        self.pad = torch.zeros(1, self.out_dim)
        if HAS_ACCELERATION:
            self.pad = self.pad.to(DEVICE)

        self.relation_embeddings = nn.Parameter(relation_embeddings)
        self.emb = RelTemporalEncoding(self.out_dim)

        self.gru = nn.GRU(input_size=self.in_dim,
                          hidden_size=self.out_dim, batch_first=True)

        self.gru.reset_parameters()

    def forward2(self, path_index, batch_relation, paths, paths_time, lengths, path_r, path_neg_index, batch_his_r):
        r_inp = self.relation_embeddings

        pad_r = torch.cat((r_inp, self.pad), dim=0)
        emb = pad_r[paths]
        emb = self.emb(emb, paths_time)

        lengths_cpu = lengths.cpu()
        packed = pack_padded_sequence(
            emb, lengths_cpu, batch_first=True, enforce_sorted=False).to(paths.device)
        _, hidden = self.gru(packed)

        path_emb = torch.cat((self.pad, hidden.squeeze(0)), dim=0)
        del emb, packed, paths

        pad_r = torch.cat((F.normalize(r_inp, dim=1),
                          self.pad.to(r_inp.device)), dim=0)
        path_emb = F.normalize(path_emb, dim=1)

        scores = torch.mm(path_emb, pad_r[batch_relation].t()).t()
        mask = torch.zeros((scores.size(0), scores.size(1))).to(scores.device)
        m_index = min(path_index.size(1), mask.size(1))
        mask = mask.scatter(1, path_index[:, 0:m_index], 1)
        max_score, max_id = torch.max(scores * mask, 1)

        scores_r = torch.mm(pad_r, pad_r.t())[batch_relation]
        his_score = torch.mean(torch.diagonal(
            scores_r[:, batch_his_r], dim1=0, dim2=1).t(), 1)

        score = max_score

        return score, path_emb[path_neg_index], pad_r[path_r]

    def test(self, path_index, batch_relation, paths, lengths, paths_time, batch_his_r):
        r_inp = self.relation_embeddings

        pad = torch.zeros(1, self.out_dim)

        pad_r = torch.cat((r_inp, pad.to(r_inp.device)), dim=0)
        emb = pad_r[paths]
        emb = self.emb(emb, paths_time)
        lengths_cpu = lengths.cpu()
        packed = pack_padded_sequence(
            emb, lengths_cpu, batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        path_emb = torch.cat(
            (self.pad.to(r_inp.device), hidden.squeeze(0)), dim=0)

        del emb, packed, paths
        pad_r = torch.cat((F.normalize(r_inp, dim=1),
                          pad.to(r_inp.device)), dim=0)
        path_emb = F.normalize(path_emb, dim=1)

        scores = torch.mm(
            path_emb, pad_r[batch_relation[0]].unsqueeze(1)).squeeze(1)
        max_score, max_id = torch.max(scores[path_index], 1)

        scores_r = torch.mm(
            pad_r, pad_r[batch_relation[0]].unsqueeze(1)).squeeze(1)
        his_score = torch.mean(scores_r[batch_his_r], 1)

        score = max_score + his_score

        print(his_score, max_id)
        return score

    def test_with_interpretability(self, path_index, batch_relation, paths, lengths, paths_time, batch_his_r):
        r_inp = self.relation_embeddings

        pad = torch.zeros(1, self.out_dim)

        pad_r = torch.cat((r_inp, pad.to(r_inp.device)), dim=0)
        emb = pad_r[paths]
        emb = self.emb(emb, paths_time)
        lengths_cpu = lengths.cpu()
        packed = pack_padded_sequence(
            emb, lengths_cpu, batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        path_emb = torch.cat(
            (self.pad.to(r_inp.device), hidden.squeeze(0)), dim=0)

        del emb, packed, paths
        pad_r = torch.cat((F.normalize(r_inp, dim=1),
                          pad.to(r_inp.device)), dim=0)
        path_emb = F.normalize(path_emb, dim=1)

        scores = torch.mm(
            path_emb, pad_r[batch_relation[0]].unsqueeze(1)).squeeze(1)
        max_score, max_id = torch.max(scores[path_index], 1)

        scores_r = torch.mm(
            pad_r, pad_r[batch_relation[0]].unsqueeze(1)).squeeze(1)
        his_score = torch.mean(scores_r[batch_his_r], 1)

        score = max_score + his_score

        return {
            'scores': score,
            'best_paths_idx': max_id,
            'his_scores': his_score,
            'path_scores': max_score
        }

    def __repr__(self):
        return self.__class__.__name__


In [109]:
id2relation = dict()
id2entity = dict()

with open(os.path.join(args.data, 'relation2id.txt'), 'r') as f:
    for line in f:
        line_split = line.split()
        rel_id = int(line_split[-1])
        relation = "_".join(line_split[:-1])
        id2relation[rel_id] = relation

with open(os.path.join(args.data, 'entity2id.txt'), 'r') as f:
    for line in f:
        line_split = line.split()
        rel_id = int(line_split[-1])
        relation = "_".join(line_split[:-1])
        id2entity[rel_id] = relation

with open(os.path.join(args.data, 'stat.txt'), 'r') as fr:
        for line in fr:
            line_split = line.split()
            _, num_r = int(line_split[0]), int(line_split[1])
    
_, _, all_triples, _ = build_data(args.data, num_r)
    

def get_entities_and_paths_info(t, r, quad_id, ps, graph_test, sorted_indices_tail=None, sorted_scores_tail=None, max_ids=None, his_scores=None, path_scores=None, top_k=10):
    def get_relation_from_id(relation_id):
        if relation_id in id2relation:
            relation = id2relation[relation_id]
        else:
            relation = f"ReverseOf({id2relation[relation_id-len(id2relation)]})"
        return relation
    
    triple = all_triples[quad_id]
    subject_entity_id, relation_id, object_entity_id = triple
    subject_entity = id2entity[subject_entity_id]
    object_entity = id2entity[object_entity_id]
    relation = get_relation_from_id(relation_id)
    
    target_entities_id = graph_test.t_r_id_target_dict[t][r][quad_id]
    target_entities = list(map(lambda entity_id: id2entity[entity_id], target_entities_id))
    correct_target = target_entities[0]
    assert object_entity == correct_target, f"Incorrect label detected. expected: {object_entity} found: {correct_target}"
    
    correct_rank = None
    if sorted_indices_tail is not None:
        correct_rank = int(torch.where(sorted_indices_tail == 0)[0][0]) + 1    
    correct_target_score = sorted_scores_tail[np.where(sorted_indices_tail.cpu().numpy() == 0)[0][0]] # type: ignore
    paths_data = graph_test.t_paths[t]

    top_predictions = []
    if sorted_indices_tail is not None:
        for rank, entity_idx in enumerate(sorted_indices_tail[:top_k]):
            entity_idx = int(entity_idx)
            if entity_idx >= len(target_entities_id):
                continue
            
            entity_id = target_entities_id[entity_idx]
            entity_name = id2entity[entity_id]
            score = sorted_scores_tail[rank].item() if sorted_scores_tail is not None else None
            is_correct = (entity_idx == 0)
            
            best_path = None
            interaction_score = 0.0
            best_path_score = None
            
            if max_ids is not None and rank < len(max_ids):
                path_idx_in_group = max_ids[rank].item()
                if entity_idx < len(ps) and path_idx_in_group < len(ps[entity_idx]):
                    best_path_id = ps[entity_idx][path_idx_in_group]
                    if best_path_id < len(paths_data):
                        relations_sequence_id = paths_data[best_path_id]
                        relations_sequence = list(map(get_relation_from_id, relations_sequence_id))
                        best_path = " -> ".join(relations_sequence)
            
            if his_scores is not None and rank < len(his_scores):
                interaction_score = his_scores[rank].item()
            
            if path_scores is not None and rank < len(path_scores):
                best_path_score = path_scores[rank].item() 

            top_predictions.append({
                'rank': rank + 1,
                'entity_name': entity_name,
                'entity_id': entity_id,
                'score': score,
                'is_correct': is_correct,
                'best_path': best_path,
                'best_path_score': best_path_score,
                'interaction_score': interaction_score
            })
    
    correct_target_extra = None
    if correct_rank and correct_rank > top_k:
        correct_idx = 0
        best_path_score = None 
        best_path = None
        interaction_score = 0.0
        
        if max_ids is not None and len(max_ids) > 0:
            correct_position = int(torch.where(sorted_indices_tail == 0)[0][0])
            if correct_position < len(max_ids):
                path_idx_in_group = max_ids[correct_position].item()
                if path_idx_in_group < len(ps[correct_idx]):
                    best_path_id = ps[correct_idx][path_idx_in_group]
                    if best_path_id < len(paths_data):
                        relations_sequence_id = paths_data[best_path_id]
                        relations_sequence = list(map(get_relation_from_id, relations_sequence_id))
                        best_path = " -> ".join(relations_sequence)
        
        if his_scores is not None:
            correct_position = int(torch.where(sorted_indices_tail == 0)[0][0])
            if correct_position < len(his_scores):
                interaction_score = his_scores[correct_position].item()

        if path_scores is not None and best_path is not None:
            correct_position = int(torch.where(sorted_indices_tail == 0)[0][0])
            if correct_position < len(path_scores):
                best_path_score = path_scores[correct_position].item()

        correct_target_extra = {
            'rank': correct_rank,
            'entity_name': correct_target,
            'score': correct_target_score.item() if correct_target_score is not None else None,
            'best_path': best_path,
            'interaction_score': interaction_score,
            'best_path_score': best_path_score,
        }    
    
    return {
        'subject_entity': subject_entity,
        'relation': relation,
        'time': t,
        'target_entities': target_entities,
        'correct_target': correct_target,
        'correct_target_score': correct_target_score,
        'correct_target_extra': correct_target_extra, 
        'correct_rank': correct_rank,
        'top_predictions': top_predictions,
        'total_candidates': len(target_entities_id) if target_entities_id else 0,
    }

def print_reasoning_analysis(reasoning_info): 
    print("="*80)
    print(f"Query: ({reasoning_info['subject_entity']}, {reasoning_info['relation']}, ?, t_{reasoning_info['time']})")
    print(f"Correct Answer: {reasoning_info['correct_target']}")
    print(f"Correct Answer Rank: {reasoning_info['correct_rank']}")
    print()
    
    print("TOP PREDICTIONS:")
    print("-"*60)
    for pred in reasoning_info['top_predictions']:
        status = "✓ CORRECT" if pred['is_correct'] else "✗ Wrong"
        score_str = f"{pred['score']:.4f}" if pred['score'] is not None else "N/A"
        path_str = pred['best_path'] if pred['best_path'] else "No path"
        interaction_str = f"{pred['interaction_score']:.4f}"
        best_path_score = f"{pred['best_path_score']:.4f}" if pred['best_path_score'] is not None else "N/A"
        
        print(f"Rank {pred['rank']:2d}: {pred['entity_name']:<30} Score: {score_str} {status}")
        print(f"         Path: {path_str} Score: {best_path_score}")
        print(f"         Interaction: {interaction_str}")
        print()

    if reasoning_info['correct_target_extra']:
        extra = reasoning_info['correct_target_extra']
        score_str = f"{extra['score']:.4f}" if extra['score'] is not None else "N/A"
        path_str = extra['best_path'] if extra['best_path'] else "No path"
        interaction_str = f"{extra['interaction_score']:.4f}"
        best_path_score = f"{extra['best_path_score']:.4f}" if extra['best_path_score'] is not None else 0.0
        
        print(f"Rank {extra['rank']:2d}: {extra['entity_name']:<30} Score: {extra['interaction_score']+best_path_score:.4f} ✓ CORRECT")
        print(f"         Path: {path_str} Score: {best_path_score:.4f}")
        print(f"         Interaction: {interaction_str}")
        print()
    
    print("="*80)
    print()

number of triples -> 100590


In [ ]:
if not is_in_colab():
    model_state_file = './results/bestmodel/ICEWS14_forecasting/his_len_13'
else:
    model_state_file = './ILR-IR/results/bestmodel/ICEWS14_forecasting/his_len_13' 
    
class RenameUnpickler(dill.Unpickler):
    def find_class(self, module, name):
        renamed_module = module
        if module == "GPT_GNN.data" or module == 'data':
            renamed_module = "pyHGT.data"
        return super(RenameUnpickler, self).find_class(renamed_module, name)


def renamed_load(file_obj):
    return RenameUnpickler(file_obj).load()

def list_to_array(x, pad):
    dff = pd.concat([pd.DataFrame({'{}'.format(index): labels})
                    for index, labels in enumerate(x)], axis=1)
    return dff.fillna(pad).values.T.astype(int)

def sample_outputs(args, time_samples=3, relation_samples=3, quad_per_relation_samples=3):
    model = TypeGAT(num_e, num_r*2, relation_embeddings, args.embedding_size)
    model.load_state_dict(torch.load(
        '{0}/trained99.pth'.format(model_state_file), map_location='cpu'), strict=False)
    model.to('cpu')
    model.eval()

    graph_test = renamed_load(
            open(os.path.join(args.data, 'graph_preprocess_test.pk'), 'rb'))

    t_r_id_p_dict_sample = {
        t: {
            r_id: {
                quad_id: graph_test.t_r_id_p_dict[t][r_id][quad_id]
                for quad_id in random.sample(
                    list(graph_test.t_r_id_p_dict[t][r_id].keys()),
                    min(quad_per_relation_samples, len(graph_test.t_r_id_p_dict[t][r_id]))
                )
            }
            for r_id in random.sample(
                list(graph_test.t_r_id_p_dict[t].keys()),
                min(relation_samples, len(graph_test.t_r_id_p_dict[t]))
            )
        }
        for t in random.sample(
                list(graph_test.t_r_id_p_dict.keys()), 
                min(time_samples, len(graph_test.t_r_id_p_dict))
            )
    }
    
    with torch.no_grad():
        for t, r_dict in t_r_id_p_dict_sample.items():
            size = 0
            for r, id_p in r_dict.items():
                for id, ps in id_p.items():
                    len_r = 0
                    batch_paths_id = []
                    batch_relation = []

                    batch_his_r = []

                    size = size + 1
                    len_r = len_r + len(ps)
                    batch_paths_id.extend(ps)

                    batch_relation.extend([r] * len_r)

                    batch_his_r.extend(graph_test.r_copy[t][r][id])

                    batch_paths_id = torch.LongTensor(
                        list_to_array(batch_paths_id, 0))
                    batch_relation = torch.LongTensor(np.array(batch_relation))

                    batch_his_r = torch.LongTensor(
                        list_to_array(batch_his_r, num_r * 2))

                    paths = graph_test.t_paths[t]
                    lengths = graph_test.t_paths_len[t]
                    paths_time = graph_test.t_paths_time[t]

                    if len(paths) != 0:
                        paths = pad_sequence([torch.LongTensor(np.array(p)) for p in paths], batch_first=True,
                                             padding_value=num_r*2)
                        paths_time = pad_sequence([torch.LongTensor(np.array(p)) for p in paths_time], batch_first=True,
                                                  padding_value=0)
                    else:
                        paths = torch.LongTensor(np.array(paths))
                        paths_time = torch.LongTensor(np.array(paths_time))
                    lengths = torch.LongTensor(np.array(lengths))

                    batch_paths_id = Variable(batch_paths_id)
                    batch_relation = Variable(batch_relation)
                    paths = Variable(paths)
                    lengths = Variable(lengths)
                    paths_time = Variable(paths_time)
                    batch_his_r = Variable(torch.LongTensor(batch_his_r))

                    inference_result = model.test_with_interpretability(
                        batch_paths_id, batch_relation, paths, lengths, paths_time, batch_his_r)
                    scores_tail, max_ids, his_scores, path_scores = inference_result['scores'], inference_result['best_paths_idx'], \
                        inference_result['his_scores'], inference_result['path_scores']
                    del batch_paths_id, batch_relation, paths, lengths

                    sorted_scores_tail, sorted_indices_tail = torch.sort(
                        scores_tail.view(-1), dim=-1, descending=True)
                    del scores_tail

                    top_k = 10
                    top_max_ids = max_ids[sorted_indices_tail[:top_k]]
                    top_his_scores = his_scores[sorted_indices_tail[:top_k]]
                    top_path_scores = path_scores[sorted_indices_tail[:top_k]]

                    reasoning_info = get_entities_and_paths_info(
                        t, r, id, ps, graph_test, 
                        sorted_indices_tail=sorted_indices_tail, 
                        sorted_scores_tail=sorted_scores_tail,
                        max_ids=top_max_ids,
                        his_scores=top_his_scores,
                        path_scores=top_path_scores,
                        top_k=10
                    )
                    
                    print_reasoning_analysis(reasoning_info)

sample_outputs(args)

Query: (Pdea, Arrest,_detain,_or_charge_with_legal_action, ?, t_348)
Correct Answer: Criminal_(Philippines)
Correct Answer Rank: 1

TOP PREDICTIONS:
------------------------------------------------------------
Rank  1: Criminal_(Philippines)         Score: 0.9882 ✓ CORRECT
         Path: ReverseOf(Praise_or_endorse) Score: 0.9882
         Interaction: 0.0000

Rank  2: Military_(Philippines)         Score: 0.4415 ✗ Wrong
         Path: ReverseOf(Return,_release_person(s)) Score: 0.4415
         Interaction: 0.0000

Rank  3: Military_Personnel_-_Special_(Philippines) Score: 0.3758 ✗ Wrong
         Path: ReverseOf(Abduct,_hijack,_or_take_hostage) Score: 0.3758
         Interaction: 0.0000

Rank  4: City_Mayor_(Philippines)       Score: 0.3188 ✗ Wrong
         Path: ReverseOf(Provide_humanitarian_aid) Score: 0.3188
         Interaction: 0.0000

Rank  5: Director_General_(Philippines) Score: 0.3174 ✗ Wrong
         Path: ReverseOf(Engage_in_negotiation) Score: 0.3174
         Interaction: 0